In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ACQ_E.xpt"
csv_file = DATA_DIR / "ACQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Preserve untouched XPT for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 ACQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ACD010A",
    "ACD010B",
    "ACD010C",
    "ACD040"
]


if df.shape != (7173, 5):
    raise ValueError(
        f"Unexpected ACQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "ACQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: ACQ_E structure = "
    "7,173 rows x 5 columns."
)


# ============================================================
# 4. IMPORTANT CODEBOOK NOTE
# ============================================================

#
# CDC HTML contains:
#
# ACQ005 - CHECK ITEM
#
# But ACQ005 is NOT a released variable in ACQ_E.xpt.
#
# DO NOT add ACQ005 to the CSV.
#

if "ACQ005" in df.columns:
    raise ValueError(
        "Unexpected ACQ005 column found."
    )


print(
    "PASS: ACQ005 correctly absent from released XPT."
)


# ============================================================
# 5. KNOWN SAS/XPORT TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY VALUE CHECK ---"
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifact found. "
        "Do NOT automatically replace with zero."
    )


print(
    "PASS: ACQ_E contains NO tiny XPORT artifacts."
)


# ============================================================
# 6. TRUE NUMERIC ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    true_zero_total
)


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .to_string()
    )

    raise ValueError(
        "Unexpected numeric zero found."
    )


print(
    "PASS: Original ACQ_E contains no numeric zeros."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print(
    "\n--- SEQN CHECK ---"
)


seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:

    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:

    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:

    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):

    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "PASS: SEQN = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 8. GENUINE DECIMAL / FRACTIONAL VALUE CHECK
# ============================================================

#
# ACQ_E should contain NO genuine decimal variables.
#
# All five released fields are identifiers/codes.
#

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:

        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:
    raise ValueError(
        "STOP: Unexpected fractional values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 9. CDC CHECK - ACD010A
# ============================================================

#
# CDC:
#
# 1  English      = 4,817
# 77 Refused      = 1
# 99 Don't know   = 0
# Missing         = 2,355
#

EXPECTED_ACD010A = {
    1: 4817,
    77: 1
}


actual = (
    df["ACD010A"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_ACD010A:

    raise ValueError(
        "ACD010A frequency mismatch.\n"
        f"Expected: {EXPECTED_ACD010A}\n"
        f"Actual:   {actual}"
    )


if int(
    df["ACD010A"]
    .isna()
    .sum()
) != 2355:

    raise ValueError(
        "ACD010A missing count differs."
    )


if int(
    df["ACD010A"]
    .eq(99)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected ACD010A=99 value."
    )


print(
    "\nPASS: ACD010A matches CDC."
)

print(
    "      1 English = 4,817"
)

print(
    "      77 Refused = 1"
)

print(
    "      Missing = 2,355"
)


# ============================================================
# 10. CDC CHECK - ACD010B
# ============================================================

#
# CDC:
#
# 8 Spanish = 6
# Missing   = 7,167
#

EXPECTED_ACD010B = {
    8: 6
}


actual = (
    df["ACD010B"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_ACD010B:

    raise ValueError(
        "ACD010B frequency mismatch.\n"
        f"Expected: {EXPECTED_ACD010B}\n"
        f"Actual:   {actual}"
    )


if int(
    df["ACD010B"]
    .isna()
    .sum()
) != 7167:

    raise ValueError(
        "ACD010B missing count differs."
    )


print(
    "PASS: ACD010B matches CDC."
)

print(
    "      8 Spanish = 6"
)

print(
    "      Missing = 7,167"
)


# ============================================================
# 11. CDC CHECK - ACD010C
# ============================================================

#
# CDC:
#
# 9 Other  = 261
# Missing  = 6,912
#

EXPECTED_ACD010C = {
    9: 261
}


actual = (
    df["ACD010C"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_ACD010C:

    raise ValueError(
        "ACD010C frequency mismatch.\n"
        f"Expected: {EXPECTED_ACD010C}\n"
        f"Actual:   {actual}"
    )


if int(
    df["ACD010C"]
    .isna()
    .sum()
) != 6912:

    raise ValueError(
        "ACD010C missing count differs."
    )


print(
    "PASS: ACD010C matches CDC."
)

print(
    "      9 Other = 261"
)

print(
    "      Missing = 6,912"
)


# ============================================================
# 12. CDC CHECK - ACD040
# ============================================================

#
# CDC:
#
# 1 Only Spanish               = 766
# 2 More Spanish than English  = 324
# 3 Both equally               = 352
# 4 More English than Spanish  = 360
# 5 Only English               = 358
# 7 Refused                    = 0
# 9 Don't know                 = 2
# Missing                      = 5,011
#

EXPECTED_ACD040 = {
    1: 766,
    2: 324,
    3: 352,
    4: 360,
    5: 358,
    9: 2
}


actual = (
    df["ACD040"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


actual = {
    int(k): int(v)
    for k, v in actual.items()
}


if actual != EXPECTED_ACD040:

    raise ValueError(
        "ACD040 frequency mismatch.\n"
        f"Expected: {EXPECTED_ACD040}\n"
        f"Actual:   {actual}"
    )


if int(
    df["ACD040"]
    .isna()
    .sum()
) != 5011:

    raise ValueError(
        "ACD040 missing count differs."
    )


if int(
    df["ACD040"]
    .eq(7)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected ACD040=7 value."
    )


print(
    "\nPASS: ACD040 matches CDC."
)

print(
    "      1 Only Spanish              = 766"
)

print(
    "      2 More Spanish than English = 324"
)

print(
    "      3 Both equally              = 352"
)

print(
    "      4 More English than Spanish = 360"
)

print(
    "      5 Only English              = 358"
)

print(
    "      9 Don't know                = 2"
)

print(
    "      Missing                     = 5,011"
)


# ============================================================
# 13. EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# IMPORTANT:
#
# ACQ_E needs NO numeric correction.
#
# Tiny artifacts = 0
# Zero restorations = 0
#
# Therefore expected_df is EXACTLY
# the original released XPT.
#

expected_df = source_df.copy()


# ============================================================
# 14. CLEAN WHOLE-NUMBER FORMATTING
# ============================================================

#
# Every variable has been proven to contain only
# whole-number identifiers/codes.
#
# Therefore:
#
# SEQN:
# 41475.0 -> 41475
#
# ACD010A:
# 1.0 -> 1
#
# ACD040:
# 3.0 -> 3
#
# Missing:
# NaN -> blank
#

for col in EXPECTED_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All ACQ_E variables converted "
    "to clean whole-number formatting."
)


# ============================================================
# 15. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal original XPT exactly."
)


# ============================================================
# 16. EXPORT CSV
# ============================================================

#
# No float_format.
# No rounding.
# Missing values become blank cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 17. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 18. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure matches XPT."
)


# ============================================================
# 19. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 5 columns numerically "
    "match original XPT exactly."
)


# ============================================================
# 20. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += differences


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing positions changed:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 21. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)

    raw_rows = list(reader)


if len(raw_rows) != 7173:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 22. STRICT RAW CSV FORMAT FIDELITY
# ============================================================

#
# Every nonmissing value should be written as:
#
# str(int(source_value))
#
# Examples:
#
# 41475.0 -> "41475"
# 1.0     -> "1"
# 77.0    -> "77"
#
# Missing -> ""
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )


    for example in raw_examples:
        print(example)


    raise ValueError(
        "ACQ_E raw CSV formatting differs."
    )


print(
    "PASS: Raw CSV tokens match expected "
    "whole-number source representation."
)


# ============================================================
# 23. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 24. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 25. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 26. FINAL CDC FREQUENCY CHECK AFTER EXPORT
# ============================================================

final_acd010a = (
    csv_df["ACD010A"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


final_acd010a = {
    int(k): int(v)
    for k, v in final_acd010a.items()
}


if final_acd010a != EXPECTED_ACD010A:
    raise ValueError(
        "Final ACD010A frequencies changed."
    )


final_acd010b = (
    csv_df["ACD010B"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


final_acd010b = {
    int(k): int(v)
    for k, v in final_acd010b.items()
}


if final_acd010b != EXPECTED_ACD010B:
    raise ValueError(
        "Final ACD010B frequencies changed."
    )


final_acd010c = (
    csv_df["ACD010C"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


final_acd010c = {
    int(k): int(v)
    for k, v in final_acd010c.items()
}


if final_acd010c != EXPECTED_ACD010C:
    raise ValueError(
        "Final ACD010C frequencies changed."
    )


final_acd040 = (
    csv_df["ACD040"]
    .value_counts(
        dropna=True
    )
    .sort_index()
    .to_dict()
)


final_acd040 = {
    int(k): int(v)
    for k, v in final_acd040.items()
}


if final_acd040 != EXPECTED_ACD040:
    raise ValueError(
        "Final ACD040 frequencies changed."
    )


print(
    "\nPASS: Final ACQ_E frequencies "
    "still match CDC/XPT."
)


# ============================================================
# 27. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("ACQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nTiny artifacts:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    true_zero_total
)

print(
    "Genuine decimal variables:",
    0
)


print(
    "\nCDC CHECK ITEM ACQ005:"
)

print(
    "Present in HTML documentation only"
)

print(
    "Absent from released XPT"
)

print(
    "Not added to CSV"
)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0 values:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "ACQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 ACQ_E XPT -> CSV
Rows: 7,173
Columns: 5
PASS: ACQ_E structure = 7,173 rows x 5 columns.
PASS: ACQ005 correctly absent from released XPT.

--- ORIGINAL XPT TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: ACQ_E contains NO tiny XPORT artifacts.

True numeric zeros: 0
PASS: Original ACQ_E contains no numeric zeros.

--- SEQN CHECK ---
PASS: SEQN = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
Genuine fractional variables: {}
PASS: Genuine decimal variables = 0.

PASS: ACD010A matches CDC.
      1 English = 4,817
      77 Refused = 1
      Missing = 2,355
PASS: ACD010B matches CDC.
      8 Spanish = 6
      Missing = 7,167
PASS: ACD010C matches CDC.
      9 Other = 261
      Missing = 6,912

PASS: ACD040 matches CDC.
      1 Only Spanish              = 766
      2 More Spanish than English = 324
      3 Both equally              = 352
      4 More English than Spanish = 360
      5 Only English              = 358
   